# Data Acquisition

In this lab, we will explore the process of data acquisition. 

In the case of *passive* network traffic analysis, there are generally two primary ways of acquiring data:
* Packet capture
* Network traffic flows (sometimes called IPFIX)

The advent of more programmability in networks is quickly changing this landscape. 

In particular, systems like Retina are now making it possible to ask more complex questions of network traffic from passive traffic capture and analysis but the general underlying traffic patterns are still based on raw packet capture.

## Background

Because packet captures are so large, it can sometimes be convenient to work with summary statistics about network traffic. Instead of the raw packets, data could represent the total number of bytes, packets, and so forth for flows. 

Raw traffic capture is thus sometimes represented as summaries of flow statistics, rather than raw packet traces. In this activity, we will *generate* the summary statistics and then think about what types of information is (and is not) available in a packet trace summary vs. a raw packet capture.

## Step 1: Load a Packet Trace

Load the packet capture from the last assignment.

In [29]:
import pandas as pd

ndf = pd.read_csv("data/netflix.csv.gz")
ndf.head(20)

,No.,Time,Source,Destination,Protocol,Length,Info
0,1,2018-02-11 08:10:00.534682,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,77,Standard query 0xed0c A fonts.gstatic.com
1,2,2018-02-11 08:10:00.534832,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,77,Standard query 0x301a AAAA fonts.gstatic.com
2,3,2018-02-11 08:10:00.539408,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,87,Standard query 0x11d3 A googleads.g.doubleclic...
3,4,2018-02-11 08:10:00.541204,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,87,Standard query 0x1284 AAAA googleads.g.doublec...
4,5,2018-02-11 08:10:00.545785,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,78,Standard query 0x3432 AAAA ytimg.l.google.com
5,6,2018-02-11 08:10:00.547036,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,96,Standard query 0xb756 A r4---sn-gxo5uxg-jqbe.g...
6,7,2018-02-11 08:10:00.547156,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,75,Standard query 0x62ab A ssl.gstatic.com
7,8,2018-02-11 08:10:00.547249,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,74,Standard query 0x42fb A www.google.com
8,9,2018-02-11 08:10:00.853950,ns-vip-pro.paris.inria.fr,192.168.43.72,DNS,386,Standard query response 0x11d3 A 216.58.213.162
9,10,2018-02-11 08:10:00.853970,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,75,Standard query 0x8756 A www.gstatic.com


## Step 2: Generate Statistics for Each Flow

A **flow** is defined as groups of packets that share the following attributes:
* Source IP Address
* Destination IP Address
* Source Port
* Destination Port
* Time interval

The csv file we used in the past assignments do not have port numbers, so you can simply group on source and destination IP address.

For each flow in the packet trace, generate the following statistics for each flow:

* Number of bytes
* Number of packets
* Duration (time)

In [34]:
ndf['Time'] = pd.to_datetime(ndf['Time'])

summary = ndf.groupby(['Source', 'Destination']).agg(
    packets=('Length', 'count'),
    bytes=('Length', 'sum'),
    start=('Time', 'min'),
    end=('Time', 'max'),
).reset_index()

summary['duration'] = (summary['end'] - summary['start']).dt.total_seconds()
summary

,Source,Destination,packets,bytes,start,end,duration
0,0.0.0.0,255.255.255.255,8,3032,2018-02-11 08:10:37.716265,2018-02-11 08:18:04.797274,447.081009
1,0.0.0.0,all-systems.mcast.net,4,184,2018-02-11 08:10:52.052108,2018-02-11 08:17:08.306861,376.254753
2,104.31.113.215,192.168.43.72,6,934,2018-02-11 08:18:13.613768,2018-02-11 08:18:16.299097,2.685329
3,17.188.166.20,192.168.43.72,2,218,2018-02-11 08:14:10.039478,2018-02-11 08:14:17.674745,7.635267
4,17.252.44.15,192.168.43.72,3,420,2018-02-11 08:11:53.050694,2018-02-11 08:16:10.425198,257.374504
...,...,...,...,...,...,...,...
72,par10s29-in-f3.1e100.net,192.168.43.72,8,1091,2018-02-11 08:18:12.636628,2018-02-11 08:18:16.298848,3.662220
73,par10s29-in-f4.1e100.net,192.168.43.72,79,71268,2018-02-11 08:10:01.240865,2018-02-11 08:14:09.701096,248.460231
74,par10s38-in-f13.1e100.net,192.168.43.72,30,7683,2018-02-11 08:10:05.894009,2018-02-11 08:14:08.881174,242.987165
75,par10s38-in-f3.1e100.net,192.168.43.72,144,98271,2018-02-11 08:10:01.209115,2018-02-11 08:18:16.313540,495.104425


### Total Number of Flows

Count the total number of flows in this trace.

77 flows

### Number of Bytes

Count the total number of bytes for each flow in the trace. 

Then, sort the flows by size, in bytes.  

What do you notice about the large flows? What do they look like?

In [36]:
# Total number of bytes
print(f"Total number of bytes: {summary['bytes'].sum()}")

#sort by size in bytes
summary = summary.sort_values('bytes', ascending=False)
summary.head(10)


Total number of bytes: 134630034


,Source,Destination,packets,bytes,start,end,duration
67,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,192.168.43.72,80084,120607242,2018-02-11 08:10:20.958591,2018-02-11 08:18:16.298969,475.340378
66,ipv4-c069-cdg001-ix.1.oca.nflxvideo.net,192.168.43.72,4873,7138148,2018-02-11 08:10:20.957155,2018-02-11 08:18:16.299010,475.341855
25,192.168.43.72,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,47902,3357228,2018-02-11 08:10:20.820787,2018-02-11 08:18:16.299106,475.478319
44,a23-57-80-120.deploy.static.akamaitechnologies...,192.168.43.72,1005,1332086,2018-02-11 08:10:03.629125,2018-02-11 08:16:16.525640,372.896515
65,ipv4-c063-cdg001-ix.1.oca.nflxvideo.net,192.168.43.72,338,431178,2018-02-11 08:11:01.246971,2018-02-11 08:12:16.340228,75.093257
46,ec2-52-19-39-146.eu-west-1.compute.amazonaws.com,192.168.43.72,472,348141,2018-02-11 08:10:03.671653,2018-02-11 08:17:13.743144,430.071491
19,192.168.43.72,ec2-52-19-39-146.eu-west-1.compute.amazonaws.com,489,340330,2018-02-11 08:10:02.903625,2018-02-11 08:17:13.743191,430.839566
24,192.168.43.72,ipv4-c069-cdg001-ix.1.oca.nflxvideo.net,3170,244455,2018-02-11 08:10:20.811333,2018-02-11 08:18:16.299106,475.487773
37,198.38.120.137,192.168.43.72,101,116255,2018-02-11 08:10:05.890178,2018-02-11 08:11:13.213402,67.323224
75,par10s38-in-f3.1e100.net,192.168.43.72,144,98271,2018-02-11 08:10:01.209115,2018-02-11 08:18:16.313540,495.104425


I notice that the larger flows have much longer durations. Their sources are typically not bit IP addresses but weblinks. 

### Number of Packets

Count the number of packets in each flow. 

What do you notice about these flows? Are they similar to the largest flows by bytes? Which differ?

I notice that these flows have alot of variance as ~5000 packets and yield ~7 million bytes which does not linearly align with the row under it, where ~48k packets yielded ~3 million bytes. However, there is an upward trend where the more packets a flow has, the more bytes that it has. 

### Duration

Compute the duration of each flow, by taking the time of the last packet and subtracting the time of the first, for each flow.  

What are the longest flows in the trace?

## Bytes and Packets Per Second

Compute the bytes per second and packets per second for each flow.

For a simple feature computation, compute the average bytes and packets per second for each flow, for the entire duration of the flow.  If you want to get more clever or fancy, you can do "windowed averages", computing bytes or packets per second for shorter time intervals.

## Note

Some of the libraries that we will use in this class, including the `netml` library from the University of Chicago, will compute these and other statistics automatically.

## Thought Questions

1. What are the largest flows in terms of: Number of bytes? Number of packets?

2. What do you notice about the flow sizes and the directions of flows?

3. What kinds of features are *not* available in packet summary statistics like those above which might be available in a raw packet trace? How might those features be useful for different packet classification problems?